# Milestone 2: EDA Dashboard
This notebook builds the eight required visualizations for M2 and saves them to `results/figures/` as 300 DPI PNG files.

## 1. Imports and Data Loading
Load the M1 output panel and parse dates for time-series analysis.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

sys.path.append(str(Path.cwd() / 'code'))
from config_paths import FIGURES_DIR, FINAL_DATA_DIR
from panel_format_utils import load_panel_as_wide

sns.set_style('whitegrid')
sns.set_palette('colorblind')
plt.rcParams['font.size'] = 11

df = load_panel_as_wide(FINAL_DATA_DIR / 'analysis_panel.csv')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Shape:', df.shape)
print('Date range:', df['date'].min().date(), 'to', df['date'].max().date())
df.head()

## 2. Summary Statistics

In [ ]:
summary = df.describe().T
summary[['mean','std','min','max']]

## 3. Correlation Analysis

In [ ]:
vars_to_plot = ['mkt_ret','sentiment_michigan_ics','bull_bear_spread','mkt_rf','smb','hml','rmw','cma','rf']
corr = df[vars_to_plot].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.5)
plt.title('M2 Plot 1: Correlation Heatmap (Returns, Sentiment, and Controls)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_01_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Returns show weak-to-moderate association with sentiment variables and stronger links with market-factor controls. This suggests M3 should include both sentiment terms and factor controls to avoid omitted-variable bias.

## 4. Time Series Visuals

In [ ]:
plt.figure(figsize=(12,4.5))
plt.plot(df['date'], df['mkt_ret'], linewidth=1.8, label='Market Return')
plt.axhline(0, color='black', linewidth=0.8, alpha=0.8)
plt.title('M2 Plot 2: Market Return Time Series (2004-2024)')
plt.xlabel('Date')
plt.ylabel('Market Return (%)')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_02_outcome_time_series.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Returns exhibit distinct volatility clustering around stress episodes, indicating non-constant variance. M3 should use robust or HAC standard errors.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,4.8))
ax1.plot(df['date'], df['mkt_ret'], color='#1f77b4', linewidth=1.8, label='Market Return')
ax1.set_xlabel('Date')
ax1.set_ylabel('Market Return (%)', color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')

ax2 = ax1.twinx()
ax2.plot(df['date'], df['sentiment_michigan_ics'], color='#d62728', linewidth=1.6, alpha=0.8, label='Michigan Sentiment')
ax2.set_ylabel('Michigan Sentiment Index', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#d62728')

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper right')
plt.title('M2 Plot 3: Dual-Axis Co-Movement (Market Return vs Michigan Sentiment)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_03_dual_axis_outcome_driver.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Co-movement exists but is not one-for-one and appears regime dependent. This supports testing lag structure and interactions instead of only contemporaneous levels.

## 5. Lagged Effect Analysis

In [ ]:
lags = [0,1,2,3,6,12]
lag_corr = []
for lag in lags:
    lag_corr.append(df['mkt_ret'].corr(df['sentiment_michigan_ics'].shift(lag)))
lag_df = pd.DataFrame({'lag_months': lags, 'correlation': lag_corr})

plt.figure(figsize=(9,4.8))
sns.barplot(data=lag_df, x='lag_months', y='correlation', color='#2ca02c')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('M2 Plot 4: Lagged Correlation (Returns vs Lagged Michigan Sentiment)')
plt.xlabel('Lag (Months)')
plt.ylabel('Correlation')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_04_lagged_effects.png', dpi=300, bbox_inches='tight')
plt.show()
lag_df

**Caption:** The largest absolute lagged correlation identifies the first lag candidate for M3. Economically, delayed reaction is plausible when sentiment changes diffuse through spending and investment behavior over several months.

## 6. Alternatives for Dataset Without Groups

In [ ]:
rolling_corr = df['mkt_ret'].rolling(window=6).corr(df['sentiment_michigan_ics'])
plt.figure(figsize=(12,4.5))
plt.plot(df['date'], rolling_corr, color='#9467bd', linewidth=1.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('M2 Plot 5: 6-Month Rolling Correlation (Returns vs Michigan Sentiment)')
plt.xlabel('Date')
plt.ylabel('Rolling Correlation')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_05_rolling_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Rolling correlations vary over time, indicating structural instability. M3 should evaluate regime effects or time-varying coefficients.

In [ ]:
periods = [('2004-2008','2004-01-01','2008-12-31'), ('2009-2014','2009-01-01','2014-12-31'), ('2015-2019','2015-01-01','2019-12-31'), ('2020-2024','2020-01-01','2024-12-31')]
rows = []
for label, start, end in periods:
    mask = (df['date'] >= pd.Timestamp(start)) & (df['date'] <= pd.Timestamp(end))
    sub = df.loc[mask]
    rows.append({'period': label, 'corr_return_sentiment': sub['mkt_ret'].corr(sub['sentiment_michigan_ics'])})
sub_df = pd.DataFrame(rows)

plt.figure(figsize=(9,4.8))
sns.barplot(data=sub_df, x='period', y='corr_return_sentiment', color='#ff7f0e')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('M2 Plot 6: Subsample Correlation by Time Period')
plt.xlabel('Period')
plt.ylabel('Correlation (Return vs Sentiment)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_06_subsample_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
sub_df

**Caption:** Subsample differences are consistent with changing macro environments and policy regimes. Interaction terms or period dummies are likely needed in M3.

## 7. Factor / Control Relationships

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,4.8))
sns.regplot(data=df, x='smb', y='mkt_ret', ax=axes[0], scatter_kws={'alpha': 0.65})
axes[0].set_title('Returns vs SMB')
axes[0].set_xlabel('SMB (%)')
axes[0].set_ylabel('Market Return (%)')
sns.regplot(data=df, x='hml', y='mkt_ret', ax=axes[1], scatter_kws={'alpha': 0.65}, color='#d62728')
axes[1].set_title('Returns vs HML')
axes[1].set_xlabel('HML (%)')
axes[1].set_ylabel('Market Return (%)')
fig.suptitle('M2 Plot 7: Factor-Control Relationships with Market Returns', y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_07_control_scatterplots.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Scatter patterns with fitted lines provide visual evidence that factor controls explain part of return variation and should remain in M3 specifications.

## 8. Time Series Decomposition

In [ ]:
series = df.set_index('date')['mkt_ret'].asfreq('ME')
decomp = seasonal_decompose(series, model='additive', period=12, extrapolate_trend='freq')
fig, axes = plt.subplots(4,1, figsize=(12,8), sharex=True)
axes[0].plot(decomp.observed)
axes[0].set_title('Observed')
axes[1].plot(decomp.trend)
axes[1].set_title('Trend')
axes[2].plot(decomp.seasonal)
axes[2].set_title('Seasonal')
axes[3].plot(decomp.resid)
axes[3].set_title('Residual')
axes[3].set_xlabel('Date')
fig.suptitle('M2 Plot 8: Time Series Decomposition of Market Returns', y=0.995)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'M2_08_time_series_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()

**Caption:** Decomposition separates long-run trend, recurring seasonal structure, and residual noise. If seasonal components are material, M3 should include seasonality controls.

## 9. File Check
Confirm all M2 figures were saved to the required output directory.

In [ ]:
sorted([p.name for p in FIGURES_DIR.glob('M2_*.png')])